# J2C-TST-IT: baseline training and generation

Runs Llama/Gemma with four-shot, LoRA, and prefix-tuning on the Italian corpus. The profile targets a 48 GB NVIDIA GPU with BF16/FlashAttention 2, 32 GB of host RAM, and early stopping with patience 1. Installation, training, and generation are enabled; rerunning the notebook resumes training from the latest checkpoint and continues generation from the missing examples.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_baselines_dir():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        for possible in (candidate, candidate / '5-baselines'):
            if (possible / 'train_adapter.py').is_file():
                return possible.resolve()
    raise FileNotFoundError('Cannot locate 5-baselines/train_adapter.py')

BASELINES_DIR = find_baselines_dir()
LANGUAGE = 'IT'
RUN_DIR = BASELINES_DIR / 'J2C-TST-IT'
OUTPUTS_ROOT = RUN_DIR / 'outputs'
PREDICTIONS_DIR = RUN_DIR / 'predictions'
INSTALL_DEPENDENCIES = True
EXECUTE_EXPENSIVE = True
FORCE_HF_LOGIN = False  # True only to replace the saved Hugging Face account/token
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
MODELS = ['llama', 'gemma']
METHODS = ['lora', 'prefix']
EARLY_STOPPING_PATIENCE = 1
PROFILE_VERSION = 'gpu48-v3-gemma-prefix-cache'
MODEL_PROFILES = {
    'llama': {'tag': 'gpu48_bf16_fa2_es1_b4', 'train_batch': 4, 'eval_batch': 8, 'grad_accum': 4, 'gen_batch': 4},
    'gemma': {'tag': 'gpu48_bf16_fa2_es1_gemma_b2', 'train_batch': 2, 'eval_batch': 4, 'grad_accum': 8, 'gen_batch': 2},
}
print('Baseline directory:', BASELINES_DIR)
print('Run directory:', RUN_DIR)
print('Profile version:', PROFILE_VERSION)
print('Gemma profile:', MODEL_PROFILES['gemma'])

In [ ]:
if INSTALL_DEPENDENCIES:
    command = [sys.executable, '-m', 'pip', 'install', '-r', str(BASELINES_DIR / 'requirements.txt')]
    print('>', subprocess.list2cmdline(command))
    subprocess.run(command, cwd=BASELINES_DIR, check=True)
    if sys.platform.startswith('linux'):
        flash_env = os.environ.copy()
        flash_env.setdefault('MAX_JOBS', '2')
        flash_env.setdefault('NVCC_THREADS', '2')
        command = [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '-r', str(BASELINES_DIR / 'requirements-flash-attn.txt')]
        print('>', subprocess.list2cmdline(command))
        subprocess.run(command, cwd=BASELINES_DIR, env=flash_env, check=True)
    elif EXECUTE_EXPENSIVE:
        raise RuntimeError('The FlashAttention 2 profile requires a Linux CUDA environment.')
    else:
        print('FlashAttention 2 installation skipped in a non-Linux dry run.')
    if EXECUTE_EXPENSIVE:
        check = "import psutil, torch, flash_attn; assert torch.cuda.is_available(), 'CUDA unavailable'; assert torch.cuda.is_bf16_supported(), 'The GPU does not support BF16'; vram=torch.cuda.get_device_properties(0).total_memory/1024**3; print(torch.cuda.get_device_name(0), f'VRAM: {vram:.1f} GiB', f'RAM: {psutil.virtual_memory().total/1024**3:.1f} GiB', 'BF16:', torch.cuda.is_bf16_supported(), 'flash-attn:', flash_attn.__version__)"
        subprocess.run([sys.executable, '-c', check], cwd=BASELINES_DIR, check=True)
else:
    print('Dependency installation skipped; set INSTALL_DEPENDENCIES=True in the configuration cell for a fresh environment.')

MODEL_REPOSITORIES = {'llama': 'meta-llama/Llama-3.2-1B', 'gemma': 'google/gemma-3-1b-it'}
if EXECUTE_EXPENSIVE:
    import inspect
    from huggingface_hub import get_token, hf_hub_download, login
    if FORCE_HF_LOGIN:
        os.environ.pop('HF_TOKEN', None)
        os.environ.pop('HUGGING_FACE_HUB_TOKEN', None)
    if FORCE_HF_LOGIN or get_token() is None:
        print('Hugging Face authentication is required. The token will not be stored in the notebook.')
        login_parameters = inspect.signature(login).parameters
        if 'skip_if_logged_in' in login_parameters:
            login(skip_if_logged_in=False)
        elif 'new_session' in login_parameters:
            login(new_session=True)
        else:
            login()
    for alias in MODELS:
        model_repository = MODEL_REPOSITORIES[alias]
        try:
            hf_hub_download(repo_id=model_repository, filename='config.json', token=get_token())
        except Exception as error:
            raise RuntimeError(
                f'Access denied to {model_repository}. Open https://huggingface.co/{model_repository}, '
                'accept its terms with the same account as the token, then rerun this cell. '
                'To switch accounts, set FORCE_HF_LOGIN=True and rerun the cell.'
            ) from error
        print('Verified Hugging Face access:', model_repository)

In [ ]:
EXPECTED_PROFILE_VERSION = 'gpu48-v3-gemma-prefix-cache'
if globals().get('PROFILE_VERSION') != EXPECTED_PROFILE_VERSION:
    raise RuntimeError('Stale kernel configuration: restart the kernel and run every cell from the beginning.')
expected_gemma = {'tag': 'gpu48_bf16_fa2_es1_gemma_b2', 'train_batch': 2, 'eval_batch': 4, 'grad_accum': 8, 'gen_batch': 2}
if MODEL_PROFILES.get('gemma') != expected_gemma:
    raise RuntimeError(f'Unsafe Gemma profile: {MODEL_PROFILES.get("gemma")!r}')
compat_check = [sys.executable, '-c', 'from train_adapter import install_gemma3_prefix_cache_compat; print("Gemma prefix compatibility: available")']
subprocess.run(compat_check, cwd=BASELINES_DIR, check=True)

def run_script(arguments):
    command = [sys.executable, *map(str, arguments)]
    print('\n>', subprocess.list2cmdline(command))
    return subprocess.run(command, cwd=BASELINES_DIR, check=True)

for model_alias in MODELS:
    profile = MODEL_PROFILES[model_alias]
    for method in METHODS:
        output_dir = OUTPUTS_ROOT / profile['tag'] / model_alias / method
        args = ['train_adapter.py', '--language', LANGUAGE, '--model-id', model_alias,
                '--method', method, '--output-dir', output_dir,
                '--train-batch-size', profile['train_batch'], '--eval-batch-size', profile['eval_batch'],
                '--gradient-accumulation-steps', profile['grad_accum'],
                '--early-stopping-patience', EARLY_STOPPING_PATIENCE,
                '--early-stopping-threshold', '0.0',
                '--bf16', '--tf32', '--attention-implementation', 'auto',
                '--dataloader-num-workers', '2', '--group-by-length',
                '--optim', 'adamw_torch_fused', '--no-full-determinism', '--resume']
        if not EXECUTE_EXPENSIVE:
            args.extend(['--max-train-samples', '2', '--max-validation-samples', '2', '--dry-run'])
        run_script(args)

In [ ]:
for model_alias in MODELS:
    profile = MODEL_PROFILES[model_alias]
    few_shot_output = PREDICTIONS_DIR / f"{LANGUAGE}_{model_alias}_few_shot_{profile['tag']}.csv"
    args = ['generate_baseline.py', '--language', LANGUAGE, '--strategy', 'few_shot',
            '--model-id', model_alias, '--output', few_shot_output,
            '--batch-size', profile['gen_batch'], '--bf16',
            '--attention-implementation', 'auto', '--resume']
    if not EXECUTE_EXPENSIVE:
        args.extend(['--limit', '2', '--dry-run'])
    run_script(args)
    for method in METHODS:
        adapter = OUTPUTS_ROOT / profile['tag'] / model_alias / method
        output = PREDICTIONS_DIR / f"{LANGUAGE}_{model_alias}_{method}_{profile['tag']}.csv"
        args = ['generate_baseline.py', '--language', LANGUAGE, '--strategy', method,
                '--model-id', model_alias, '--adapter-path', adapter, '--output', output,
                '--batch-size', profile['gen_batch'], '--bf16',
                '--attention-implementation', 'auto', '--resume']
        if not EXECUTE_EXPENSIVE:
            args.extend(['--limit', '2', '--dry-run'])
        run_script(args)

The six complete Italian prediction files are written to `J2C-TST-IT/predictions/` and can be validated by the Italian notebook in `6-evaluation/J2C-TST-IT`.